# 🏥 Medical Insurance Cost Prediction

**Predicting individual medical insurance charges using Machine Learning**

[![Python](https://img.shields.io/badge/Python-3.8+-blue.svg)](https://python.org)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.0+-orange.svg)](https://scikit-learn.org)

### Project Overview
This project builds and evaluates multiple regression models to predict medical insurance costs based on patient demographics and health factors. The best model is saved for deployment via a Streamlit web application.

### Dataset Features
| Feature | Type | Description |
|---------|------|-------------|
| `age` | Numerical | Age of the primary beneficiary |
| `sex` | Categorical | Gender (male/female) |
| `bmi` | Numerical | Body Mass Index |
| `children` | Numerical | Number of dependents covered |
| `smoker` | Categorical | Smoking status (yes/no) |
| `region` | Categorical | US residential region |
| `charges` | Numerical | **Target** — Individual medical costs billed |


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

os.makedirs('models', exist_ok=True)
print('Libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('data/insurance.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
print('=== Statistical Summary ===')
df.describe()

In [ ]:
print('=== Categorical Columns Distribution ===')
for col in ['sex', 'smoker', 'region']:
    print(f'\n{col.upper()}:')
    print(df[col].value_counts())
    print(f'Proportions:\n{df[col].value_counts(normalize=True).round(3)}')

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribution of Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
axes[0,0].hist(df['age'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Count')

# BMI distribution
axes[0,1].hist(df['bmi'], bins=30, color='darkorange', edgecolor='white', alpha=0.85)
axes[0,1].axvline(18.5, color='green', linestyle='--', label='Underweight <18.5')
axes[0,1].axvline(24.9, color='blue', linestyle='--', label='Normal <24.9')
axes[0,1].axvline(29.9, color='red', linestyle='--', label='Overweight <29.9')
axes[0,1].set_title('BMI Distribution', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('BMI')
axes[0,1].legend(fontsize=8)

# Charges distribution
axes[1,0].hist(df['charges'], bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1,0].set_title('Insurance Charges Distribution', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Charges (USD)')

# Charges log-transformed
axes[1,1].hist(np.log1p(df['charges']), bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1,1].set_title('Log-Transformed Charges', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Log(Charges)')

plt.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('models/eda_distributions.png', bbox_inches='tight')
plt.show()
print('Charges are right-skewed — log transformation makes it more normal.')

### 3.2 Categorical Features Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, palette in zip(axes, ['sex','smoker','region'],
                             ['Set2','Set1','Set3']):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values,
           color=sns.color_palette(palette, len(counts)))
    ax.set_title(f'{col.capitalize()} Distribution', fontweight='bold')
    ax.set_xlabel(col.capitalize())
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/eda_categorical.png', bbox_inches='tight')
plt.show()

### 3.3 Impact of Features on Insurance Charges

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Charges vs Age
axes[0,0].scatter(df['age'], df['charges'],
    c=df['smoker'].map({'yes':'red','no':'steelblue'}),
    alpha=0.5, edgecolors='none')
axes[0,0].set_xlabel('Age'); axes[0,0].set_ylabel('Charges (USD)')
axes[0,0].set_title('Age vs Charges (Red=Smoker)', fontweight='bold')

# Charges vs BMI
axes[0,1].scatter(df['bmi'], df['charges'],
    c=df['smoker'].map({'yes':'red','no':'steelblue'}),
    alpha=0.5, edgecolors='none')
axes[0,1].set_xlabel('BMI'); axes[0,1].set_ylabel('Charges (USD)')
axes[0,1].set_title('BMI vs Charges (Red=Smoker)', fontweight='bold')

# Smoker vs Charges
df.boxplot(column='charges', by='smoker', ax=axes[0,2],
    boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[0,2].set_title('Smoker vs Charges', fontweight='bold')
axes[0,2].set_xlabel('Smoker'); axes[0,2].set_ylabel('Charges (USD)')
plt.sca(axes[0,2]); plt.title('Smoker vs Charges', fontweight='bold')

# Region vs Charges
df.boxplot(column='charges', by='region', ax=axes[1,0],
    boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[1,0].set_xlabel('Region'); axes[1,0].set_ylabel('Charges (USD)')
plt.sca(axes[1,0]); plt.title('Region vs Charges', fontweight='bold')

# Children vs Charges
df.boxplot(column='charges', by='children', ax=axes[1,1],
    boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[1,1].set_xlabel('Number of Children'); axes[1,1].set_ylabel('Charges (USD)')
plt.sca(axes[1,1]); plt.title('Children vs Charges', fontweight='bold')

# Sex vs Charges
df.boxplot(column='charges', by='sex', ax=axes[1,2],
    boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[1,2].set_xlabel('Sex'); axes[1,2].set_ylabel('Charges (USD)')
plt.sca(axes[1,2]); plt.title('Sex vs Charges', fontweight='bold')

fig.suptitle('')
plt.suptitle('Feature Impact on Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/eda_feature_impact.png', bbox_inches='tight')
plt.show()
print('KEY INSIGHT: Smoking status has the most dramatic impact on insurance charges!')

### 3.4 Correlation Heatmap

In [ ]:
df_encoded = df.copy()
df_encoded['sex'] = df_encoded['sex'].map({'male':0,'female':1})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes':1,'no':0})
df_encoded['region'] = df_encoded['region'].map({'northeast':0,'northwest':1,'southeast':2,'southwest':3})

plt.figure(figsize=(10, 8))
corr = df_encoded.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('models/correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('Correlation with Charges:')
print(corr['charges'].sort_values(ascending=False))

## 4. Feature Engineering

In [ ]:
df_model = df.copy()

# BMI categories
df_model['bmi_category'] = pd.cut(
    df_model['bmi'],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=[0, 1, 2, 3]  # underweight, normal, overweight, obese
).astype(int)

# Age groups
df_model['age_group'] = pd.cut(
    df_model['age'],
    bins=[0, 25, 40, 55, 100],
    labels=[0, 1, 2, 3]  # young, adult, middle-aged, senior
).astype(int)

# Smoker-BMI interaction (high risk: smoker AND obese)
df_model['smoker_encoded'] = df_model['smoker'].map({'yes':1,'no':0})
df_model['bmi_smoker'] = df_model['bmi'] * df_model['smoker_encoded']

# Encode categorical columns
df_model['sex'] = df_model['sex'].map({'male':0,'female':1})
df_model['smoker'] = df_model['smoker'].map({'yes':1,'no':0})
df_model['region'] = df_model['region'].map({'northeast':0,'northwest':1,'southeast':2,'southwest':3})

df_model = df_model.drop(columns=['smoker_encoded'])

print('Feature engineered dataset shape:', df_model.shape)
print('New features added: bmi_category, age_group, bmi_smoker')
df_model.head()

## 5. Data Preparation

In [ ]:
X = df_model.drop(columns=['charges'])
y = df_model['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Testing set  : {X_test.shape[0]} samples')
print(f'Features     : {list(X.columns)}')

## 6. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)

    train_r2   = r2_score(y_tr, y_pred_train)
    test_r2    = r2_score(y_te, y_pred_test)
    test_mae   = mean_absolute_error(y_te, y_pred_test)
    test_rmse  = np.sqrt(mean_squared_error(y_te, y_pred_test))
    test_mape  = np.mean(np.abs((y_te - y_pred_test) / y_te)) * 100

    # 5-fold cross-validation
    cv_scores = cross_val_score(model, X_tr, y_tr,
                                cv=KFold(n_splits=5, shuffle=True, random_state=42),
                                scoring='r2')

    return {
        'Model'       : name,
        'Train R²'    : round(train_r2, 4),
        'Test R²'     : round(test_r2, 4),
        'CV R² Mean'  : round(cv_scores.mean(), 4),
        'CV R² Std'   : round(cv_scores.std(), 4),
        'MAE (USD)'   : round(test_mae, 2),
        'RMSE (USD)'  : round(test_rmse, 2),
        'MAPE (%)'    : round(test_mape, 2),
    }, model, y_pred_test

print('Evaluation function ready.')

In [ ]:
results = []
trained_models = {}
predictions     = {}

models = [
    ('Linear Regression',     LinearRegression(),              X_train_scaled, X_test_scaled),
    ('Ridge Regression',      Ridge(alpha=10),                 X_train_scaled, X_test_scaled),
    ('Lasso Regression',      Lasso(alpha=1),                  X_train_scaled, X_test_scaled),
    ('Decision Tree',         DecisionTreeRegressor(max_depth=8, random_state=42), X_train, X_test),
    ('Random Forest',         RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), X_train, X_test),
    ('Gradient Boosting',     GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                  max_depth=4, random_state=42), X_train, X_test),
]

print('Training models...\n')
for name, model, Xtr, Xte in models:
    res, fitted_model, y_pred = evaluate_model(name, model, Xtr, Xte, y_train, y_test)
    results.append(res)
    trained_models[name] = (fitted_model, Xtr is X_train_scaled)
    predictions[name]    = y_pred
    print(f'✔ {name} — Test R²: {res["Test R²"]:.4f}, RMSE: ${res["RMSE (USD)"]:,.0f}')

results_df = pd.DataFrame(results).set_index('Model')
print('\nAll models trained!')

### 6.1 Model Comparison Table

In [ ]:
print('=== Model Performance Comparison ===')
display(results_df.style
    .highlight_max(subset=['Test R²','CV R² Mean'], color='#c6efce')
    .highlight_min(subset=['MAE (USD)','RMSE (USD)','MAPE (%)'], color='#c6efce')
    .highlight_max(subset=['Train R²'], color='#ffeb9c')
    .format({'Train R²': '{:.4f}', 'Test R²': '{:.4f}',
             'CV R² Mean': '{:.4f}', 'CV R² Std': '{:.4f}',
             'MAE (USD)': '${:,.0f}', 'RMSE (USD)': '${:,.0f}', 'MAPE (%)': '{:.1f}%'}))

### 6.2 Model Comparison Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_list = results_df.index.tolist()
colors = ['#e74c3c','#f39c12','#f1c40f','#2ecc71','#3498db','#9b59b6']

# R² comparison
x = np.arange(len(models_list))
w = 0.35
axes[0].bar(x-w/2, results_df['Train R²'], w, label='Train R²', color='steelblue', alpha=0.8)
axes[0].bar(x+w/2, results_df['Test R²'],  w, label='Test R²',  color='darkorange', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('R² Score'); axes[0].set_title('R² Score Comparison', fontweight='bold')
axes[0].legend(); axes[0].set_ylim(0, 1.05)

# RMSE
bars = axes[1].bar(models_list, results_df['RMSE (USD)'], color=colors, alpha=0.85)
axes[1].set_xticks(range(len(models_list)))
axes[1].set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('RMSE (USD)'); axes[1].set_title('RMSE Comparison (Lower is Better)', fontweight='bold')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'${bar.get_height():,.0f}', ha='center', fontsize=8)

# MAPE
bars2 = axes[2].bar(models_list, results_df['MAPE (%)'], color=colors, alpha=0.85)
axes[2].set_xticks(range(len(models_list)))
axes[2].set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
axes[2].set_ylabel('MAPE (%)'); axes[2].set_title('MAPE Comparison (Lower is Better)', fontweight='bold')
for bar in bars2:
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/model_comparison.png', bbox_inches='tight')
plt.show()

## 7. Best Model Deep Dive

In [ ]:
best_model_name = results_df['Test R²'].idxmax()
print(f'Best Model: {best_model_name}')
print(f'Test R²   : {results_df.loc[best_model_name, "Test R²"]:.4f}')
print(f'RMSE      : ${results_df.loc[best_model_name, "RMSE (USD)"]:,.2f}')
print(f'MAE       : ${results_df.loc[best_model_name, "MAE (USD)"]:,.2f}')
print(f'MAPE      : {results_df.loc[best_model_name, "MAPE (%)"]:.2f}%')

### 7.1 Actual vs Predicted Plot

In [ ]:
best_preds = predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test, best_preds, alpha=0.5, color='steelblue', edgecolors='none', s=30)
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges (USD)', fontsize=12)
axes[0].set_ylabel('Predicted Charges (USD)', fontsize=12)
axes[0].set_title(f'Actual vs Predicted — {best_model_name}', fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.5, color='darkorange', edgecolors='none', s=30)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Charges (USD)', fontsize=12)
axes[1].set_ylabel('Residuals (USD)', fontsize=12)
axes[1].set_title('Residual Plot', fontweight='bold')

plt.suptitle(f'Best Model Analysis: {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/actual_vs_predicted.png', bbox_inches='tight')
plt.show()

### 7.2 Feature Importance

In [ ]:
best_model_obj = trained_models[best_model_name][0]

if hasattr(best_model_obj, 'feature_importances_'):
    importances = best_model_obj.feature_importances_
    feat_names  = X.columns.tolist()
    feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
    feat_df = feat_df.sort_values('Importance', ascending=True)

    plt.figure(figsize=(10, 6))
    colors_imp = ['#3498db' if i != len(feat_df)-1 else '#e74c3c'
                  for i in range(len(feat_df))]
    bars = plt.barh(feat_df['Feature'], feat_df['Importance'],
                    color=colors_imp, edgecolor='white')
    plt.xlabel('Feature Importance Score', fontsize=12)
    plt.title(f'Feature Importance — {best_model_name}', fontsize=14, fontweight='bold')
    for bar in bars:
        plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('models/feature_importance.png', bbox_inches='tight')
    plt.show()

    print('Top 3 Most Important Features:')
    print(feat_df.sort_values('Importance', ascending=False).head(3).to_string(index=False))
elif hasattr(best_model_obj, 'coef_'):
    coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': best_model_obj.coef_})
    coef_df['Abs_Coeff'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values('Abs_Coeff', ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(coef_df['Feature'], coef_df['Coefficient'],
             color=['#e74c3c' if x < 0 else '#2ecc71' for x in coef_df['Coefficient']])
    plt.xlabel('Coefficient Value')
    plt.title('Linear Model Coefficients', fontweight='bold')
    plt.tight_layout()
    plt.savefig('models/feature_importance.png', bbox_inches='tight')
    plt.show()

## 8. Hyperparameter Tuning (Best Model)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

print('Running RandomizedSearchCV on Gradient Boosting...')

param_dist = {
    'n_estimators'   : randint(100, 400),
    'learning_rate'  : uniform(0.05, 0.2),
    'max_depth'      : randint(3, 7),
    'min_samples_split': randint(2, 10),
    'subsample'      : uniform(0.7, 0.3),
}

gb_model = GradientBoostingRegressor(random_state=42)
random_search = RandomizedSearchCV(
    gb_model, param_dist, n_iter=30, cv=5,
    scoring='r2', n_jobs=-1, random_state=42, verbose=0
)
random_search.fit(X_train, y_train)

print(f'Best Parameters: {random_search.best_params_}')
print(f'Best CV R²     : {random_search.best_score_:.4f}')

tuned_model = random_search.best_estimator_
y_tuned_pred = tuned_model.predict(X_test)
tuned_r2   = r2_score(y_test, y_tuned_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_tuned_pred))
tuned_mae  = mean_absolute_error(y_test, y_tuned_pred)

print(f'\nTuned Model Test R²  : {tuned_r2:.4f}')
print(f'Tuned Model RMSE     : ${tuned_rmse:,.2f}')
print(f'Tuned Model MAE      : ${tuned_mae:,.2f}')

## 9. Save Best Model for Deployment

In [ ]:
# Save the tuned model and scaler
joblib.dump(tuned_model, 'models/best_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# Save feature names for inference
feature_names = list(X.columns)
import json
with open('models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('Model saved to models/best_model.pkl')
print('Scaler saved to models/scaler.pkl')
print('Feature names:', feature_names)

# Verify loading
loaded_model = joblib.load('models/best_model.pkl')
test_pred = loaded_model.predict(X_test[:5])
print('\nModel load verification — first 5 predictions:')
print([f'${p:,.2f}' for p in test_pred])

## 10. Interactive Prediction System

In [ ]:
import pandas as pd

def predict_insurance_cost(age, sex, bmi, children, smoker, region):
    """
    Predict medical insurance cost for a new patient.

    Parameters:
    - age     : int   (18-64)
    - sex     : str   ('male' or 'female')
    - bmi     : float (body mass index)
    - children: int   (0-5)
    - smoker  : str   ('yes' or 'no')
    - region  : str   ('northeast','northwest','southeast','southwest')
    """
    sex_enc    = 1 if sex == 'female' else 0
    smoker_enc = 1 if smoker == 'yes' else 0
    region_enc = {'northeast':0,'northwest':1,'southeast':2,'southwest':3}[region]

    bmi_cat = 0 if bmi < 18.5 else 1 if bmi < 24.9 else 2 if bmi < 29.9 else 3
    age_grp = 0 if age < 25 else 1 if age < 40 else 2 if age < 55 else 3
    bmi_smoker = bmi * smoker_enc

    feature_names = list(X.columns)
    features = pd.DataFrame([[age, sex_enc, bmi, children, smoker_enc,
                               region_enc, bmi_cat, age_grp, bmi_smoker]],
                             columns=feature_names)
    prediction = tuned_model.predict(features)[0]
    return max(0, prediction)


# --- Example Predictions ---
examples = [
    {'age':25, 'sex':'male',   'bmi':22.0, 'children':0, 'smoker':'no',  'region':'northeast'},
    {'age':45, 'sex':'female', 'bmi':35.0, 'children':2, 'smoker':'yes', 'region':'southeast'},
    {'age':60, 'sex':'male',   'bmi':28.5, 'children':3, 'smoker':'no',  'region':'southwest'},
]

print('=== Insurance Cost Predictions ===' )
print(f'{"Patient":<10}{"Age":<6}{"Sex":<10}{"BMI":<8}{"Kids":<6}{"Smoker":<8}{"Region":<14}{"Predicted Cost"}')
print('-'*75)
for i, p in enumerate(examples, 1):
    cost = predict_insurance_cost(**p)
    print(f'{i:<10}{p["age"]:<6}{p["sex"]:<10}{p["bmi"]:<8}{p["children"]:<6}{p["smoker"]:<8}{p["region"]:<14}${cost:>12,.2f}')

## 11. Key Insights & Conclusions

In [ ]:
print('=== PROJECT SUMMARY ===')
print(f'\nDataset          : {df.shape[0]} records, {df.shape[1]} features')
print(f'Best Model       : {best_model_name} (with hyperparameter tuning)')
print(f'Test R²          : {tuned_r2:.4f} ({tuned_r2*100:.1f}% variance explained)')
print(f'RMSE             : ${tuned_rmse:,.2f}')
print(f'MAE              : ${tuned_mae:,.2f}')
print()
print('=== KEY INSIGHTS ===')
print('1. SMOKING is the #1 driver — smokers pay 3-4x more in premiums')
print('2. AGE has a strong positive linear correlation with charges')
print('3. BMI > 30 (obesity) significantly increases costs, especially for smokers')
print('4. REGION has minimal impact on insurance charges')
print('5. Gradient Boosting outperforms linear models due to non-linear relationships')
print('6. Feature engineering (bmi_smoker interaction) improved model accuracy')
print()
print('=== MODEL RANKING (by Test R²) ===')
print(results_df['Test R²'].sort_values(ascending=False).to_string())